# Intersectional fairness audit — UCI Adult

A walkthrough of the `bias_audit` package: train an income classifier, then score every
populated `Gender x Race x Age` subgroup against the `Male_White_Middle-aged (30-50)` baseline.

Run from the repository root. The dataset downloads once and is cached to `data/`.


In [ ]:
import sys

sys.path.insert(0, '..')  # so the package imports when the notebook runs from notebooks/

import pandas as pd

from bias_audit import DEFAULT_CONFIG, run_audit, build_report
from bias_audit.plots import intersectional_heatmap, single_vs_intersectional, top_disadvantaged_bar

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)


## 1. Run the audit

`run_audit` loads and cleans the data, splits it, fits logistic regression, and scores every
subgroup. Protected attributes travel with the split, so predictions and demographics stay aligned.


In [ ]:
run = run_audit(config=DEFAULT_CONFIG)

run.model.performance


## 2. Which subgroups are worst off?

`spd` is the gap in approval rate against the baseline; `dir` is the ratio the four-fifths rule
applies to. `n_positives` matters: EOD is a TPR gap, so a subgroup with only a few positive
labels gets `eod_reliable = False`.


In [ ]:
run.audit.worst('spd', k=10)[
    ['intersectional_group', 'n_samples', 'n_positives', 'selection_rate', 'spd', 'dir', 'eod', 'eod_reliable']
]


In [ ]:
failing = run.audit.failing_four_fifths()
print(f'{len(failing)} of {run.audit.n_evaluated} scored subgroups breach the four-fifths rule')
print(f'{len(run.audit.results) - run.audit.n_evaluated} suppressed for small sample size')


## 3. What the single-attribute audit misses

The same predictions, scored one attribute at a time. This is the conventional analysis.


In [ ]:
run.audit.single_attribute[['attribute', 'group', 'n_samples', 'spd', 'dir', 'eod']]


`masking_summary` contrasts the worst single-attribute value with the worst subgroup *inside*
that same attribute level. The `hidden_gap` column is the disadvantage a conventional audit
would never report.


In [ ]:
run.masking


In [ ]:
single_vs_intersectional(run.audit, metric='spd', k=10)


## 4. Figures


In [ ]:
intersectional_heatmap(run.audit.results, metric='spd')


In [ ]:
top_disadvantaged_bar(run.audit.results, metric='dir', k=10)


## 5. Full report

Every number below is read out of the results table.


In [ ]:
print(build_report(run.audit, run.model.performance, run.masking))


## 6. Persist

Writes the CSV tables, figures and report into `results/`. The CLI does the same thing:
`python main.py`.


In [ ]:
from bias_audit import write_results
from bias_audit.plots import save_all_figures

for path in write_results(run) + save_all_figures(run.audit, config=run.config):
    print(path)
